# Fixation mRNN Inter-Regional Current Geometry

Train or load the derivative weight 5 / curvature weight 3 model for 100K iterations, check reconstruction quality, then inspect how source-to-target recurrent currents align with within-condition and cross-condition latent progression directions.

## 1. Setup

In [ ]:
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = next(parent for parent in Path.cwd().parents if (parent / "src").exists())

import sys
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

from dal_monte_2022_analysis.ephys.modeling import (
    backproject_replay_outputs_to_firing_rates,
    extract_fixation_latent_dynamics,
    extract_region_current_vectors,
    load_fixation_mrnn_config,
    make_targets,
    pc_reconstructed_firing_rate_accuracy,
    reconstruction_accuracy,
    replay_fixation_mrnn_run,
    resolve_fixation_mrnn_output_root,
    settings_from_config,
    train_fixation_mrnn_scratch,
)

plt.rcParams.update({"figure.dpi": 140, "axes.spines.top": False, "axes.spines.right": False})

## 2. Model Settings

In [ ]:
cfg = load_fixation_mrnn_config(repo_root / "configs/ephys_fixation_mrnn.yaml")

settings = settings_from_config(
    cfg,
    overrides={
        "target_mode": "region_pcs",
        "temporal_basis_count": 0,
        "hidden_units": 50,
        "lr": 1e-3,
        "epochs": 100_000,
        "initialization_mode": "single",
        "device": "auto",
        "temporal_derivative_loss_scale": 5.0,
        "temporal_curvature_loss_scale": 3.0,
        "correlation_loss_scale": 0.0,
        "variance_loss_scale": 0.0,
        "fr_reconstruction_loss_scale": 0.0,
        "fr_temporal_derivative_loss_scale": 0.0,
        "fr_temporal_curvature_loss_scale": 0.0,
    },
)

run_mode = "load"  # use "train" to fit missing checkpoints
overwrite = False
scratch_id = "interregional_current_d5_c3_50u_100k"
output_root = resolve_fixation_mrnn_output_root(settings) / "scratch"
run_dir = output_root / scratch_id
targets = make_targets(settings)
timeline = np.asarray(targets.timeline_s, dtype=float)

pd.DataFrame(
    [
        {
            "scratch_id": scratch_id,
            "hidden_units_per_region": settings.hidden_units,
            "lr": settings.lr,
            "epochs": settings.epochs,
            "temporal_derivative_loss_scale": settings.temporal_derivative_loss_scale,
            "temporal_curvature_loss_scale": settings.temporal_curvature_loss_scale,
            "correlation_loss_scale": settings.correlation_loss_scale,
            "variance_loss_scale": settings.variance_loss_scale,
            "fr_reconstruction_loss_scale": settings.fr_reconstruction_loss_scale,
        }
    ]
)

## 3. Fit or Load Replay

In [ ]:
checkpoint_path = run_dir / "checkpoint_final.pth"
if run_mode == "train" or not checkpoint_path.exists():
    result = train_fixation_mrnn_scratch(settings, scratch_id=scratch_id, overwrite=overwrite)
    run_dir = Path(result["run_dir"])

history = pd.read_csv(run_dir / "history.csv")
replay = replay_fixation_mrnn_run(run_dir, device=settings.device)
condition_order = tuple(replay["condition_order"])
region_order = tuple(replay["region_order"])
latent = extract_fixation_latent_dynamics(replay)
current_vectors = extract_region_current_vectors(replay)

pd.DataFrame([{"run_dir": str(run_dir), "iterations": int(history["iteration"].iloc[-1])}])

## 4. Loss Trajectory

In [ ]:
weighted_columns = {
    "reconstruction_loss": 1.0,
    "temporal_derivative_loss": settings.temporal_derivative_loss_scale,
    "temporal_curvature_loss": settings.temporal_curvature_loss_scale,
    "correlation_loss": settings.correlation_loss_scale,
    "variance_loss": settings.variance_loss_scale,
    "fr_reconstruction_loss": settings.fr_reconstruction_loss_scale,
    "fr_temporal_derivative_loss": settings.fr_temporal_derivative_loss_scale,
    "fr_temporal_curvature_loss": settings.fr_temporal_curvature_loss_scale,
}

fig, ax = plt.subplots(figsize=(7.5, 3.2))
for column, weight in weighted_columns.items():
    if float(weight) == 0.0 or column not in history.columns:
        continue
    ax.plot(history["iteration"], history[column] * float(weight), linewidth=1.0, label=f"{column} x {weight:g}")
ax.plot(history["iteration"], history["loss"], color="black", linewidth=1.5, label="total loss")
ax.set(title="d5 / c3 model", xlabel="iteration", ylabel="weighted loss")
ax.set_yscale("log")
ax.legend(frameon=False, fontsize=7)
fig.tight_layout()

## 5. Final Fit Quality

In [ ]:
final_losses = pd.DataFrame(
    [
        {
            "loss": float(history["loss"].iloc[-1]),
            "reconstruction_loss": float(history["reconstruction_loss"].iloc[-1]),
            "temporal_derivative_loss": float(history["temporal_derivative_loss"].iloc[-1]),
            "temporal_curvature_loss": float(history["temporal_curvature_loss"].iloc[-1]),
        }
    ]
)
metrics = pd.concat(
    [
        reconstruction_accuracy(replay).assign(metric_space="region_pcs"),
        pc_reconstructed_firing_rate_accuracy(replay).assign(metric_space="backprojected_fr"),
    ],
    ignore_index=True,
)
fit_quality = (
    metrics.groupby(["metric_space", "region"], as_index=False)
    .agg(mean_mse=("mse", "mean"), mean_mae=("mae", "mean"), mean_r2=("r2", "mean"), mean_corr=("correlation", "mean"))
    .sort_values(["metric_space", "region"])
)
display(final_losses)
display(fit_quality)

## 6. Random-Region PC Reconstruction

In [ ]:
rng = np.random.default_rng()
pc_region = str(rng.choice(region_order))
n_pcs = min(6, targets.pcs_by_region[pc_region].shape[-1])

fig, axes = plt.subplots(n_pcs, len(condition_order), figsize=(3.2 * len(condition_order), 1.9 * n_pcs), sharex=True, squeeze=False)
for pc_idx in range(n_pcs):
    for cond_col, condition in enumerate(condition_order):
        ax = axes[pc_idx, cond_col]
        cond_idx = condition_order.index(condition)
        ax.plot(timeline, targets.pcs_by_region[pc_region][cond_idx, :, pc_idx], color="black", linewidth=2.0, label="target")
        yhat = replay["output_by_region"][pc_region].detach().cpu().numpy()[cond_idx, :, pc_idx]
        ax.plot(timeline, yhat, color="#2f6fbb", linewidth=1.2, label="model")
        ax.axvline(0.0, color="0.5", linewidth=0.7)
        if pc_idx == 0:
            ax.set_title(condition)
        if cond_col == 0:
            ax.set_ylabel(f"PC{pc_idx + 1}")
        if pc_idx == n_pcs - 1:
            ax.set_xlabel("time (s)")
axes[0, -1].legend(frameon=False, fontsize=7)
fig.suptitle(f"Region: {pc_region}", y=1.01)
fig.tight_layout()

## 7. Random-Region Backprojected Firing Rates

In [ ]:
fr_region = str(rng.choice(region_order))
target_fr_by_region = targets.pc_reconstructed_raw_by_region()
predicted_fr = backproject_replay_outputs_to_firing_rates(replay)[fr_region]
n_units = target_fr_by_region[fr_region].shape[-1]
unit_indices = np.sort(rng.choice(n_units, size=min(6, n_units), replace=False))

fig, axes = plt.subplots(len(unit_indices), len(condition_order), figsize=(3.2 * len(condition_order), 1.9 * len(unit_indices)), sharex=True, squeeze=False)
for row, unit_idx in enumerate(unit_indices):
    for cond_col, condition in enumerate(condition_order):
        ax = axes[row, cond_col]
        cond_idx = condition_order.index(condition)
        ax.plot(timeline, target_fr_by_region[fr_region][cond_idx, :, unit_idx], color="black", linewidth=2.0, label="target")
        ax.plot(timeline, predicted_fr[cond_idx, :, unit_idx], color="#2f6fbb", linewidth=1.2, label="model")
        ax.axvline(0.0, color="0.5", linewidth=0.7)
        if row == 0:
            ax.set_title(condition)
        if cond_col == 0:
            ax.set_ylabel(f"unit {unit_idx}")
        if row == len(unit_indices) - 1:
            ax.set_xlabel("time (s)")
axes[0, -1].legend(frameon=False, fontsize=7)
fig.suptitle(f"Region: {fr_region}", y=1.01)
fig.tight_layout()

## 8. Latent Geometry Helpers

Projection convention for the current-alignment panels: for active fixation condition `c`, source region `s`, target region `r`, and reference condition `q`, compute

`direction_q(t) = W_rec h_t[q, r] - h_t[c, r]`

in target-region hidden coordinates. The source-to-target current `I_s_to_r(c,t)` is projected onto the unit-normalized `direction_q(t)`. Positive values mean the source current points toward that reference condition's `W_rec h_t` point; negative values mean it points away.

In [ ]:
condition_colors = {
    "face_interactive": "#b64198",
    "face_non_interactive": "#4c9a2a",
    "object": "#6f4e37",
}


def region_slice(region):
    start, stop = replay["model"].mrnn.get_region_indices(region)
    return slice(int(start), int(stop))


def fit_pca2(values):
    flat = np.asarray(values, dtype=float).reshape(-1, values.shape[-1])
    mean = flat.mean(axis=0, keepdims=True)
    centered = flat - mean
    _, _, vt = np.linalg.svd(centered, full_matrices=False)
    components = np.zeros((2, values.shape[-1]), dtype=float)
    n_fit = min(2, vt.shape[0])
    if n_fit:
        components[:n_fit] = vt[:n_fit]
    return mean.squeeze(0), components


def project_to_pca2(values, mean, components):
    flat = np.asarray(values, dtype=float).reshape(-1, values.shape[-1])
    scores = (flat - mean) @ components.T
    return scores.reshape(*values.shape[:-1], 2)


def recurrent_drive_by_condition_region(region):
    sl = region_slice(region)
    return np.stack([latent[condition]["recurrent_drive"][:, sl].numpy() for condition in condition_order], axis=0)


def hidden_state_by_condition_region(region):
    sl = region_slice(region)
    return np.stack([latent[condition]["hidden_state"][:, sl].numpy() for condition in condition_order], axis=0)


def nearest_time_index(time_s):
    return int(np.argmin(np.abs(timeline - float(time_s))))

## 9. `W_rec h_t` Space by Region

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8.5, 7.2), squeeze=False)
for ax, region in zip(axes.ravel(), region_order):
    drive = recurrent_drive_by_condition_region(region)
    mean, components = fit_pca2(drive)
    scores = project_to_pca2(drive, mean, components)
    for cond_idx, condition in enumerate(condition_order):
        color = condition_colors.get(condition, "0.3")
        xy = scores[cond_idx]
        centroid = xy.mean(axis=0)
        ax.plot(xy[:, 0], xy[:, 1], color=color, linewidth=1.2, label=condition)
        ax.scatter(centroid[0], centroid[1], color=color, s=45, edgecolor="black", linewidth=0.5, zorder=5)
    ax.set(title=region, xlabel="W@h PC1", ylabel="W@h PC2")
axes[0, -1].legend(frameon=False, fontsize=7)
fig.tight_layout()

## 10. Schematic: `h_t` to Same- and Other-Fixation `W_rec h_t` Points

In [ ]:
schematic_region = str(rng.choice(region_order))
active_condition = "face_interactive"
time_s = -0.2
time_idx = nearest_time_index(time_s)
active_idx = condition_order.index(active_condition)

drive = recurrent_drive_by_condition_region(schematic_region)
hidden = hidden_state_by_condition_region(schematic_region)
mean, components = fit_pca2(drive)
drive_scores = project_to_pca2(drive, mean, components)
hidden_score = project_to_pca2(hidden[active_idx : active_idx + 1, time_idx : time_idx + 1], mean, components)[0, 0]

fig, ax = plt.subplots(figsize=(6.4, 5.4))
for cond_idx, condition in enumerate(condition_order):
    color = condition_colors.get(condition, "0.3")
    xy = drive_scores[cond_idx]
    centroid = xy.mean(axis=0)
    ax.plot(xy[:, 0], xy[:, 1], color=color, alpha=0.5, linewidth=1.0)
    ax.scatter(centroid[0], centroid[1], color=color, s=55, edgecolor="black", linewidth=0.5, label=f"{condition} centroid")
    target_point = xy[time_idx]
    ax.scatter(target_point[0], target_point[1], color=color, s=35, zorder=5)
    ax.annotate(
        "",
        xy=target_point,
        xytext=hidden_score,
        arrowprops={"arrowstyle": "->", "linestyle": ":", "linewidth": 1.2, "color": color},
    )
ax.scatter(hidden_score[0], hidden_score[1], color="black", s=80, marker="x", linewidth=2.0, label=f"h_t {active_condition}")
ax.set(
    title=f"{schematic_region}: h_t at {timeline[time_idx]:.3f}s to W@h points",
    xlabel="W@h PC1",
    ylabel="W@h PC2",
)
ax.legend(frameon=False, fontsize=7)
fig.tight_layout()

## 11. Current Projection Helper

In [ ]:
def current_projection_table(active_condition):
    active_idx = condition_order.index(active_condition)
    rows = []
    for target_region in region_order:
        hidden = hidden_state_by_condition_region(target_region)
        drive = recurrent_drive_by_condition_region(target_region)
        active_hidden = hidden[active_idx]
        reference_dirs = {
            condition: drive[condition_order.index(condition)] - active_hidden
            for condition in condition_order
        }
        for source_region in region_order:
            current = current_vectors[(source_region, target_region)].numpy()[active_idx]
            for ref_condition, direction in reference_dirs.items():
                norm = np.linalg.norm(direction, axis=-1)
                unit_direction = np.divide(
                    direction,
                    np.maximum(norm[:, None], 1e-8),
                    out=np.zeros_like(direction),
                    where=norm[:, None] > 1e-8,
                )
                projection = np.sum(current * unit_direction, axis=-1)
                for time_idx, value in enumerate(projection):
                    rows.append(
                        {
                            "active_condition": active_condition,
                            "reference_condition": ref_condition,
                            "source_region": source_region,
                            "target_region": target_region,
                            "time_idx": int(time_idx),
                            "time_s": float(timeline[time_idx]),
                            "projection": float(value),
                        }
                    )
    return pd.DataFrame(rows)


def plot_current_projection_grid(active_condition):
    df = current_projection_table(active_condition)
    fig, axes = plt.subplots(len(region_order), len(region_order), figsize=(12.5, 10.5), squeeze=False, sharex=True)
    for row, target_region in enumerate(region_order):
        for col, source_region in enumerate(region_order):
            ax = axes[row, col]
            subset = df[(df["target_region"] == target_region) & (df["source_region"] == source_region)]
            ax.axhline(0.0, color="black", linewidth=0.6, alpha=0.5)
            ax.axvline(0.0, color="0.5", linewidth=0.6, alpha=0.45)
            for ref_condition in condition_order:
                trace = subset[subset["reference_condition"] == ref_condition].sort_values("time_idx")
                x = trace["time_s"].to_numpy(dtype=float)
                y = trace["projection"].to_numpy(dtype=float)
                color = condition_colors.get(ref_condition, "0.3")
                ax.plot(x, y, color=color, linewidth=1.0, label=ref_condition)
                ax.fill_between(x, 0.0, y, color=color, alpha=0.12)
            if row == 0:
                ax.set_title(f"source {source_region}", fontsize=9)
            if col == 0:
                ax.set_ylabel(f"target {target_region}\nprojection")
            if row == len(region_order) - 1:
                ax.set_xlabel("time (s)")
    axes[0, -1].legend(frameon=False, fontsize=7, loc="upper left", bbox_to_anchor=(1.02, 1.0))
    fig.suptitle(f"Active fixation type: {active_condition}", y=1.01)
    fig.tight_layout()
    return fig, axes, df

## 12. Current Projections: Face Interactive

In [ ]:
fig_face_int, axes_face_int, current_projection_face_int = plot_current_projection_grid("face_interactive")

## 13. Current Projections: Face Non-Interactive

In [ ]:
fig_face_nonint, axes_face_nonint, current_projection_face_nonint = plot_current_projection_grid("face_non_interactive")

## 14. Current Projections: Object

In [ ]:
fig_object, axes_object, current_projection_object = plot_current_projection_grid("object")